# Notebook 03 — MLForecast (LightGBM)

**ISA 444 Final Project — Retail Forecasting (Walmart)**

Mirrors the structure of `duke_energy_mlforecast.ipynb` from class.

### What this notebook does
1. Load the prepared top-20 training and test parquets.
2. Define **lag features** (1, 2, 4, 13, 52) and **rolling statistics** that LightGBM will use.
3. Wire in **exogenous regressors**: holidays, markdowns, temperature, and static store features.
4. Run **5-fold non-overlapping CV** at `h = 4` and evaluate ME / MAE / RMSE / MAPE per series.
5. Inspect **feature importance** to understand what the model leans on.
6. Refit on all training data and produce **future forecasts** for the Kaggle test dates.

### Why LightGBM fits this problem
Tree models are a great middle ground between the rigid stats baselines and the heavier neural models:
- They're **global** — one model trained on all 20 series, learning shared patterns.
- They handle **mixed feature types** (numeric lags + categorical store_type + holiday flags) natively.
- They're **fast** — full CV on this dataset takes under a minute on a laptop.
- They give you **feature importance** for free, which makes the model interpretable.

### Key project decisions in this notebook
- `lag_52` is the marquee feature for retail — it carries 'same week last year' info.
- LightGBM hyperparameters are kept modest (250 trees, depth 6, learning_rate 0.05). The Duke notebook used very similar settings.
- We pass `is_holiday_week`, MarkDown1-5, and Temperature as `dynamic` (known-in-future) regressors. Kaggle provides all of these for test dates.
- `store_type` and `store_size` are **static** features (constant per series).


## Install Packages

Run once on a fresh environment, then comment out.

In [1]:
# !pip install mlforecast lightgbm utilsforecast pyarrow

## Library Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import bias, mae, rmse, mape

## Paths and Settings

Keep these consistent with Notebooks 01 and 02 so all CV results merge cleanly later.

In [3]:
DATA_DIR = Path("../data")
OUT_DIR  = Path("../outputs")
(OUT_DIR / "cv_results").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "forecasts").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

FREQ           = "W-FRI"
H              = 4
N_WINDOWS      = 5
STEP_SIZE      = 4
RANDOM_SEED    = 42

## Load Training and Test Data

In [4]:
df_train = pd.read_parquet(DATA_DIR / "walmart_top20_train.parquet")
df_test  = pd.read_parquet(DATA_DIR / "walmart_top20_test.parquet")

print("Training rows :", len(df_train))
print("Training series:", df_train["unique_id"].nunique())
print("Test rows     :", len(df_test))
df_train.head()

Training rows : 2860
Training series: 20
Test rows     : 780


,unique_id,ds,y,is_holiday_week,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,had_markdown,Temperature,Fuel_Price,CPI,Unemployment,store_type,store_size,week_of_year,month,quarter,year
0,S10_D72,2010-02-05,232558.51,0,0.0,0.0,0.0,0.0,0.0,0,54.34,2.962,126.442065,9.765,B,126512,5,2,1,2010
1,S10_D72,2010-02-12,202622.42,1,0.0,0.0,0.0,0.0,0.0,0,49.96,2.828,126.496258,9.765,B,126512,6,2,1,2010
2,S10_D72,2010-02-19,184982.01,0,0.0,0.0,0.0,0.0,0.0,0,58.22,2.915,126.526286,9.765,B,126512,7,2,1,2010
3,S10_D72,2010-02-26,186002.43,0,0.0,0.0,0.0,0.0,0.0,0,52.77,2.825,126.552286,9.765,B,126512,8,2,1,2010
4,S10_D72,2010-03-05,191989.54,0,0.0,0.0,0.0,0.0,0.0,0,55.92,2.877,126.578286,9.765,B,126512,9,3,1,2010


## Encode the Static Categorical Feature

`store_type` is the only string column (values A / B / C). LightGBM can handle categoricals natively if we set the dtype to `category`, but it's safer and more portable to **one-hot encode** it now — that way the same encoding is applied identically to train and test data, and the columns match exactly.

After encoding, the static features that travel with each series will be `store_size`, plus whichever `store_type_*` dummies exist in our 20-series subset.


In [5]:
# We need a consistent encoding across train and test, so build the union of
# store_type values first, then encode both frames against that union.
all_store_types = sorted(
    set(df_train["store_type"].unique()) | set(df_test["store_type"].unique())
)
print("store_type values in our subset:", all_store_types)

def encode_store_type(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    # Reindex columns so train and test produce the SAME dummy columns even if
    # one frame happens to be missing a category.
    dummies = pd.get_dummies(out["store_type"], prefix="store_type")
    for st in all_store_types:
        col = f"store_type_{st}"
        if col not in dummies.columns:
            dummies[col] = 0
    dummies = dummies[[f"store_type_{st}" for st in all_store_types]].astype(int)
    return pd.concat([out.drop(columns=["store_type"]), dummies], axis=1)

df_train = encode_store_type(df_train)
df_test  = encode_store_type(df_test)

# The static feature columns that will be passed to MLForecast.
STORE_TYPE_COLS = [f"store_type_{st}" for st in all_store_types]
STATIC_FEATS    = ["store_size"] + STORE_TYPE_COLS
print("Static features the model will see:", STATIC_FEATS)

store_type values in our subset: ['A', 'B']
Static features the model will see: ['store_size', 'store_type_A', 'store_type_B']


## Define the Exogenous Feature Set

**Dynamic exog** = values known for both train and test (we'll feed them at predict time). These come straight from `features.csv`:

- `is_holiday_week` — Kaggle gives future holiday flags
- `MarkDown1..5` — markdown promo amounts
- `Temperature` — Kaggle includes future temperatures
- Calendar features (`week_of_year`, `month`, `quarter`, `year`) — trivially known in advance

Notice we drop `CPI`, `Unemployment`, `Fuel_Price` here: they move slowly enough that they add little at a 4-week horizon, and some are forward-filled in the test frame. Excluding them keeps the model honest.


In [6]:
DYNAMIC_EXOG = [
    "is_holiday_week",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "had_markdown",
    "Temperature",
    "week_of_year", "month", "quarter", "year",
]

# Training frame: must contain BOTH dynamic and static features.
# MLForecast learns from static_features at fit() time and remembers them per series.
MODEL_COLS = ["unique_id", "ds", "y"] + DYNAMIC_EXOG + STATIC_FEATS
df_train_model = df_train[MODEL_COLS].copy()

# Test/future frame: ONLY dynamic exog goes into X_df at predict time.
# Static features were already learned during fit() and MLForecast carries them
# along internally. Including static columns in X_df raises an error like:
#   "Found extra columns ['store_size', 'store_type_A', ...] in X_df."
df_test_model  = df_test[["unique_id", "ds"] + DYNAMIC_EXOG].copy()

print("Training frame for ML model :", df_train_model.shape)
print("  -> includes", len(DYNAMIC_EXOG), "dynamic exog +", len(STATIC_FEATS), "static features")
print("Future X_df for predict     :", df_test_model.shape)
print("  -> dynamic exog only; static features already learned at fit time")
df_train_model.head()

Training frame for ML model : (2860, 18)
  -> includes 12 dynamic exog + 3 static features
Future X_df for predict     : (780, 14)
  -> dynamic exog only; static features already learned at fit time


,unique_id,ds,y,is_holiday_week,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,had_markdown,Temperature,week_of_year,month,quarter,year,store_size,store_type_A,store_type_B
0,S10_D72,2010-02-05,232558.51,0,0.0,0.0,0.0,0.0,0.0,0,54.34,5,2,1,2010,126512,0,1
1,S10_D72,2010-02-12,202622.42,1,0.0,0.0,0.0,0.0,0.0,0,49.96,6,2,1,2010,126512,0,1
2,S10_D72,2010-02-19,184982.01,0,0.0,0.0,0.0,0.0,0.0,0,58.22,7,2,1,2010,126512,0,1
3,S10_D72,2010-02-26,186002.43,0,0.0,0.0,0.0,0.0,0.0,0,52.77,8,2,1,2010,126512,0,1
4,S10_D72,2010-03-05,191989.54,0,0.0,0.0,0.0,0.0,0.0,0,55.92,9,3,1,2010,126512,0,1


## Configure LightGBM

Modest hyperparameters: 250 boosting rounds, learning rate 0.05, depth 6. This matches the Duke Energy ML notebook closely.

- `n_estimators=250` — enough trees to converge on this dataset without overfitting
- `learning_rate=0.05` — standard slow-and-steady setting
- `max_depth=6` / `num_leaves=31` — limits tree complexity
- `subsample=0.8`, `colsample_bytree=0.8` — light regularization through bagging
- `min_child_samples=20` — guards against splits on tiny groups (important with only 143 weeks per series)
- `verbosity=-1` — quiet logging


In [7]:
lgbm = LGBMRegressor(
    n_estimators       = 250,
    learning_rate      = 0.05,
    max_depth          = 6,
    num_leaves         = 31,
    min_child_samples  = 20,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    random_state       = RANDOM_SEED,
    n_jobs             = 1,        # 1 thread keeps memory predictable on 8GB
    verbosity          = -1,
)

## Configure MLForecast — Lags and Rolling Transforms

Following the Duke Energy ML recipe almost verbatim, adapted to weekly Walmart data:

- **Lags 1, 2, 4, 13, 26, 52** — recent (1, 2), month (4), quarter (13), half-year (26), and **same week last year (52)** — the last one is the killer feature for retail.
- **Rolling mean (4, 13)** on `lag_1` — short-term level estimates.
- **Rolling std (13)** on `lag_1` — local volatility (spikes around holidays).
- **Rolling mean (4)** on `lag_52` — average of the four weeks around the same time last year (smooths out the lag-52 signal).


In [8]:
mlf = MLForecast(
    models = {"LightGBM": lgbm},
    freq   = FREQ,
    # Raw lag features.
    lags   = [1, 2, 4, 13, 26, 52],
    # Transforms on top of lags (rolling stats give the model trend/volatility info).
    lag_transforms = {
        1:  [RollingMean(window_size=4), RollingMean(window_size=13), RollingStd(window_size=13)],
        52: [RollingMean(window_size=4)],
    },
    # MLForecast adds these date-derived features on the fly — but since we ALREADY
    # have them in DYNAMIC_EXOG, we leave date_features empty to avoid duplication.
    date_features = [],
    num_threads   = 1,
)
mlf

MLForecast(models=[LightGBM], freq=W-FRI, lag_features=['lag1', 'lag2', 'lag4', 'lag13', 'lag26', 'lag52', 'rolling_mean_lag1_window_size4', 'rolling_mean_lag1_window_size13', 'rolling_std_lag1_window_size13', 'rolling_mean_lag52_window_size4'], date_features=[], num_threads=1)

## Cross-Validation

Same 5-fold non-overlapping setup as Notebook 02. MLForecast handles all the lag bookkeeping automatically — at each fold cutoff, it computes lags using only data up to that cutoff (no leakage).

`static_features=STATIC_FEATS` tells MLForecast which columns are constant per series. Anything else in the dataframe (beyond `unique_id, ds, y`) is treated as dynamic / time-varying.


In [9]:
%%time
cv_ml = mlf.cross_validation(
    df              = df_train_model,
    h               = H,
    n_windows       = N_WINDOWS,
    step_size       = STEP_SIZE,
    static_features = STATIC_FEATS,
)
print("CV output shape:", cv_ml.shape)
cv_ml.head()

CV output shape: (400, 5)
CPU times: total: 1.7 s
Wall time: 1.85 s


,unique_id,ds,cutoff,y,LightGBM
0,S10_D72,2012-06-15,2012-06-08,105499.39,118111.201475
1,S10_D72,2012-06-22,2012-06-08,107949.41,113732.432242
2,S10_D72,2012-06-29,2012-06-08,96579.10,113182.936943
3,S10_D72,2012-07-06,2012-06-08,100464.25,119781.663643
4,S13_D90,2012-06-15,2012-06-08,122102.95,129008.367121


In [10]:
# Save raw CV predictions for downstream plotting / inspection.
cv_ml[["unique_id", "ds", "cutoff", "y", "LightGBM"]].to_csv(
    OUT_DIR / "cv_results" / "cv_mlforecast_predictions.csv", index=False
)
print("Saved:", (OUT_DIR / "cv_results" / "cv_mlforecast_predictions.csv").resolve())

Saved: C:\Users\23mwa\outputs\cv_results\cv_mlforecast_predictions.csv


## Evaluate Per Series, Per Metric

In [11]:
eval_ml = evaluate(
    cv_ml[["unique_id", "ds", "cutoff", "y", "LightGBM"]],
    metrics = [bias, mae, rmse, mape],
    models  = ["LightGBM"],
)

eval_ml.to_csv(OUT_DIR / "cv_results" / "eval_mlforecast.csv", index=False)
print("Saved:", (OUT_DIR / "cv_results" / "eval_mlforecast.csv").resolve())
eval_ml.head(8)

Saved: C:\Users\23mwa\outputs\cv_results\eval_mlforecast.csv


,unique_id,cutoff,metric,LightGBM
0,S10_D72,2012-06-08,bias,13579.021076
1,S13_D90,2012-06-08,bias,7178.921415
2,S13_D92,2012-06-08,bias,3457.274085
3,S13_D95,2012-06-08,bias,-3559.839091
4,S14_D92,2012-06-08,bias,25374.686372
5,S14_D95,2012-06-08,bias,25062.386590
6,S19_D92,2012-06-08,bias,4004.465105
7,S1_D92,2012-06-08,bias,-2414.715769


In [12]:
# Aggregate mean per metric across all 20 series.
agg_ml = (
    eval_ml
    .drop(columns=["unique_id"])
    .groupby("metric")
    .mean()
    .round(2)
)
display(agg_ml)
agg_ml.to_csv(OUT_DIR / "cv_results" / "eval_mlforecast_aggregate.csv")

,cutoff,LightGBM
metric,,
bias,2012-08-03,4228.96
mae,2012-08-03,7872.44
mape,2012-08-03,0.06
rmse,2012-08-03,9183.86


## Feature Importance

One of the best parts of using a tree model — we can see exactly which features the model relied on. We refit a *single* LightGBM on the full training data (no CV) just to extract importance.

**What we expect to see for retail weekly data:**
- `lag_1` and the rolling means should dominate — short-term level is the strongest signal.
- `lag_52` (or `rolling_mean_lag52_window_size4`) should be high — same week last year drives holidays.
- `is_holiday_week` and the calendar features should matter for spike weeks.
- MarkDown features will probably rank lower (most are zero before Nov 2011) but not negligible.


In [13]:
%%time
# fit() trains on the full training data and stores the model internally.
mlf.fit(df=df_train_model, static_features=STATIC_FEATS)

# Pull the trained LightGBM model out of the MLForecast object.
fitted_lgbm = mlf.models_["LightGBM"]

# Get importance and the feature names MLForecast actually constructed (lags + transforms + exog + static).
feat_imp = (
    pd.DataFrame({
        "feature":    fitted_lgbm.feature_name_,
        "importance": fitted_lgbm.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(feat_imp.head(20))
feat_imp.to_csv(OUT_DIR / "cv_results" / "lightgbm_feature_importance.csv", index=False)

,feature,importance
0,lag52,515
1,week_of_year,296
2,rolling_mean_lag52_window_size4,291
3,MarkDown3,284
4,lag4,273
5,lag1,260
6,lag13,237
7,lag26,188
8,store_size,181
9,Temperature,175


CPU times: total: 344 ms
Wall time: 444 ms


## Future Forecasts For The Kaggle Test Dates

We pass `X_df=df_test_model` so MLForecast knows the exog values for each future week (holidays, markdowns, temperature, calendar). LightGBM uses those plus the lags it computes internally from training history.

Horizon = the number of weeks in the test set (39).


In [14]:
%%time
h_future = df_test_model.groupby("unique_id").size().max()
print("Future horizon (weeks):", h_future)

# X_df supplies the future values of the dynamic exog columns + static features.
fc_ml = mlf.predict(h=h_future, X_df=df_test_model)

fc_ml.to_csv(OUT_DIR / "forecasts" / "test_mlforecast.csv", index=False)
print("Saved:", (OUT_DIR / "forecasts" / "test_mlforecast.csv").resolve())
print("Shape:", fc_ml.shape)
fc_ml.head()

Future horizon (weeks): 39
Saved: C:\Users\23mwa\outputs\forecasts\test_mlforecast.csv
Shape: (780, 3)
CPU times: total: 375 ms
Wall time: 485 ms


,unique_id,ds,LightGBM
0,S10_D72,2012-11-02,130795.089309
1,S10_D72,2012-11-09,133863.750942
2,S10_D72,2012-11-16,127845.798288
3,S10_D72,2012-11-23,245804.914071
4,S10_D72,2012-11-30,158622.482001


## Notebook 03 — Done

**Outputs:**
- `..\outputs\cv_results\cv_mlforecast_predictions.csv` — every CV fold prediction
- `..\outputs\cv_results\eval_mlforecast.csv` — per-series, per-metric evaluation
- `..\outputs\cv_results\eval_mlforecast_aggregate.csv` — mean across series
- `..\outputs\cv_results\lightgbm_feature_importance.csv` — what LightGBM learned to rely on
- `..\outputs\forecasts\test_mlforecast.csv` — future forecasts on Kaggle test dates

**Next:** Notebook 04 will run AutoNBEATS and AutoNHITS via NeuralForecast — these are the heaviest models in the pipeline, so we'll be careful with hyperparameters to keep memory under 8GB.
